# Verifiche a taglio — esempio completo

Verifica SLU a taglio per una trave in calcestruzzo armato secondo NTC18.

**Schema strutturale**: trave semplicemente appoggiata, luce L = 6 m, carico uniformemente distribuito.

**Dati materiali**:
- Calcestruzzo C25/30: f_ck = 25 MPa, f_ctk = 1.8 MPa
- Acciaio B450C: f_yk = 450 MPa

**Sezione**: rettangolare 300 × 550 mm (d = 500 mm)

Il notebook segue la procedura NTC18 §4.1.2.3.5:
1. Resistenze di progetto
2. Taglio senza staffe — verifica se sufficiente
3. Staffe necessarie — progetto e verifica
4. Verifica interazione torsione-taglio (caso con momento torcente)

In [ ]:
import math
from pyntc.checks.concrete import (
    concrete_design_compressive_strength,
    concrete_design_tensile_strength,
    steel_design_strength,
    shear_resistance_no_stirrups,
    shear_resistance_with_stirrups,
    torsion_resistance,
    torsion_shear_interaction,
)

## 1. Dati geometrici e di carico

In [11]:
# Geometria sezione
bw   = 300.0   # mm  larghezza anima
h    = 550.0   # mm  altezza totale
c    = 40.0    # mm  copriferro
phi  = 10.0    # mm  diametro staffa
d    = h - c - phi - 10.0   # mm  altezza utile (barra long. Ø20 / 2 ≈ 10)
print(f"Altezza utile d = {d:.0f} mm")

# Armatura longitudinale
n_bars = 4
phi_l  = 20.0               # mm  diametro barre longitudinali
Al     = n_bars * math.pi * phi_l**2 / 4
rho_l  = Al / (bw * d)
print(f"Area armatura long. Al = {Al:.1f} mm²")
print(f"Rapporto geometrico ρl = {rho_l:.4f} ({rho_l*100:.2f} %)")

# Carico SLU di progetto
L     = 6000.0    # mm  luce
q_Ed  = 80.0      # kN/m  carico SLU distribuito
V_Ed  = q_Ed * L / 2 / 1000   # kN  taglio massimo in appoggio
V_Ed_N = V_Ed * 1e3            # N   (unità SI per pyntc)
print(f"\nTaglio di progetto V_Ed = {V_Ed:.1f} kN")

Altezza utile d = 490 mm
Area armatura long. Al = 1256.6 mm²
Rapporto geometrico ρl = 0.0085 (0.85 %)

Taglio di progetto V_Ed = 240.0 kN


## 2. Resistenze di progetto dei materiali — NTC18 §4.1.2.1

In [ ]:
# Valori attesi — formule [4.1.3], [4.1.4], [4.1.5]
f_ck  = 25.0   # MPa
f_ctk = 1.8    # MPa
f_yk  = 450.0  # MPa
expected_f_cd  = 0.85 * f_ck  / 1.5    # = 14.167 MPa
expected_f_ctd = f_ctk / 1.5           # = 1.200  MPa
expected_f_yd  = f_yk  / 1.15          # = 391.304 MPa

f_cd  = concrete_design_compressive_strength(f_ck)
f_ctd = concrete_design_tensile_strength(f_ctk)
f_yd  = steel_design_strength(f_yk)

print(f"f_cd  = {f_cd:.2f} MPa")
print(f"f_ctd = {f_ctd:.3f} MPa")
print(f"f_yd  = {f_yd:.2f} MPa")

assert math.isclose(f_cd,  expected_f_cd,  rel_tol=1e-3)
assert math.isclose(f_ctd, expected_f_ctd, rel_tol=1e-3)
assert math.isclose(f_yd,  expected_f_yd,  rel_tol=1e-3)

## 3. Taglio senza armatura trasversale — NTC18 §4.1.2.3.5.1, [4.1.23]

Per sezioni senza staffe di calcolo il taglio resistente è:

$$V_{Rd} = \max\left[\left(C_{Rd,c}\,k\,(100\,\rho_l\,f_{ck})^{1/3} + 0.15\,\sigma_{cp}\right)b_w\,d,\;(v_{min} + 0.15\,\sigma_{cp})\,b_w\,d\right]$$

con $k = 1 + \sqrt{200/d} \leq 2$.

In [ ]:
# Valori attesi — formula [4.1.23]
# k = 1 + sqrt(200/d) ≤ 2;  C_Rd,c = 0.18/γ_c
_k     = min(1.0 + math.sqrt(200.0 / d), 2.0)
_C_Rdc = 0.18 / 1.5
_v_min = 0.035 * _k**1.5 * f_ck**0.5
expected_V_Rd0_N = max(
    (_C_Rdc * _k * (100 * rho_l * f_ck)**(1/3)) * bw * d,
    _v_min * bw * d,
)

V_Rd0    = shear_resistance_no_stirrups(f_ck=f_ck, d=d, bw=bw, rho_l=rho_l, sigma_cp=0.0)
V_Rd0_kN = V_Rd0 / 1e3

print(f"Fattore di scala k       = {_k:.3f}")
print(f"V_Rd (senza staffe)      = {V_Rd0_kN:.2f} kN")
print(f"V_Ed                     = {V_Ed:.2f} kN")

if V_Ed <= V_Rd0_kN:
    print("✓  Verifica soddisfatta senza staffe di calcolo")
else:
    print(f"✗  Staffe necessarie — deficit {V_Ed - V_Rd0_kN:.2f} kN")

assert math.isclose(V_Rd0, expected_V_Rd0_N, rel_tol=1e-3)

## 4. Taglio con armatura trasversale — NTC18 §4.1.2.3.5.2, [4.1.27]–[4.1.29]

Modello a traliccio generalizzato con angolo θ variabile:

$$V_{Rsd} = 0.9\,d\,\frac{A_{sw}}{s}\,f_{yd}\,(\cot\alpha + \cot\theta)\sin\alpha$$

$$V_{Rcd} = 0.9\,d\,b_w\,\alpha_c\,\nu\,f_{cd}\,\frac{\cot\alpha + \cot\theta}{1 + \cot^2\theta}$$

$$V_{Rd} = \min(V_{Rsd},\;V_{Rcd})$$

**Scelta**: staffe verticali (α = 90°), Ø10/150 (2 branche)

In [ ]:
# Staffe Ø10/150, 2 branche
phi_sw    = 10.0
n_branche = 2
Asw       = n_branche * math.pi * phi_sw**2 / 4
s         = 150.0
cot_theta = 2.5

# Valori attesi — formule [4.1.27]-[4.1.29]; alpha=90° → sin_a=1, cot_a=0
# V_Rsd = 0.9*d*(Asw/s)*f_yd*(cot_a+cot_theta)*sin_a = 0.9*d*(Asw/s)*f_yd*cot_theta
# V_Rcd = 0.9*d*bw*ν*f_cd*(cot_a+cot_theta)/(1+cot_theta²) = 0.9*d*bw*0.5*f_cd*cot_theta/(1+cot_theta²)
_V_Rsd = 0.9 * d * (Asw / s) * f_yd * cot_theta
_V_Rcd = 0.9 * d * bw * 0.5 * f_cd * cot_theta / (1.0 + cot_theta**2)
expected_V_Rd_N = min(_V_Rsd, _V_Rcd)

print(f"Asw  = {Asw:.1f} mm²  (Ø{phi_sw:.0f} 2br.)")
print(f"s    = {s:.0f} mm")
print(f"cotθ = {cot_theta}  →  θ = {math.degrees(math.atan(1/cot_theta)):.1f}°")
print()

V_Rd    = shear_resistance_with_stirrups(
    d=d, bw=bw, Asw=Asw, s=s, f_yd=f_yd, f_cd=f_cd,
    cot_theta=cot_theta, alpha=90.0, sigma_cp=0.0,
)
V_Rd_kN = V_Rd / 1e3
eta     = V_Ed / V_Rd_kN

print(f"V_Rsd         = {_V_Rsd/1e3:.2f} kN  (lato acciaio)")
print(f"V_Rcd         = {_V_Rcd/1e3:.2f} kN  (lato cls)")
print(f"V_Rd          = {V_Rd_kN:.2f} kN  = min(V_Rsd, V_Rcd)")
print(f"η = V_Ed/V_Rd = {eta:.3f}")
print("✓  Verifica taglio soddisfatta" if eta <= 1.0 else "✗  NON soddisfatta")

assert math.isclose(V_Rd, expected_V_Rd_N, rel_tol=1e-3)

## 5. Studio parametrico — variazione del passo staffe

Determina il passo massimo delle staffe che soddisfa la verifica.

In [15]:
import numpy as np

passi = np.arange(50, 351, 25)   # mm
print(f"{'Passo s [mm]':>14}  {'V_Rd [kN]':>10}  {'η = V_Ed/V_Rd':>14}  {'OK?':>6}")
print("-" * 52)
for s_i in passi:
    V_Rd_i = shear_resistance_with_stirrups(
        d=d, bw=bw, Asw=Asw, s=float(s_i),
        f_yd=f_yd, f_cd=f_cd, cot_theta=cot_theta,
    ) / 1e3
    eta_i = V_Ed / V_Rd_i
    ok = "✓" if eta_i <= 1.0 else "✗"
    print(f"{s_i:>14.0f}  {V_Rd_i:>10.2f}  {eta_i:>14.3f}  {ok:>6}")

  Passo s [mm]   V_Rd [kN]   η = V_Ed/V_Rd     OK?
----------------------------------------------------
            50      323.15           0.743       ✓
            75      323.15           0.743       ✓
           100      323.15           0.743       ✓
           125      323.15           0.743       ✓
           150      323.15           0.743       ✓
           175      323.15           0.743       ✓
           200      323.15           0.743       ✓
           225      301.18           0.797       ✓
           250      271.06           0.885       ✓
           275      246.42           0.974       ✓
           300      225.89           1.062       ✗
           325      208.51           1.151       ✗
           350      193.62           1.240       ✗


## 6. Influenza dell'angolo θ dei puntoni — NTC18 §4.1.2.3.5.2

L'angolo ottimale minimizza l'armatura a taglio. Al variare di θ cambiano sia V_Rsd (domina acciaio) che V_Rcd (domina calcestruzzo).

In [16]:
cot_theta_vals = np.linspace(1.0, 2.5, 7)
print(f"{'cotθ':>6}  {'θ [°]':>7}  {'V_Rd [kN]':>10}  {'η':>6}")
print("-" * 35)
for ct in cot_theta_vals:
    theta_deg = math.degrees(math.atan(1.0 / ct))
    V_Rd_i = shear_resistance_with_stirrups(
        d=d, bw=bw, Asw=Asw, s=s,
        f_yd=f_yd, f_cd=f_cd, cot_theta=ct,
    ) / 1e3
    eta_i = V_Ed / V_Rd_i
    print(f"{ct:>6.2f}  {theta_deg:>7.1f}  {V_Rd_i:>10.2f}  {eta_i:>6.3f}")

  cotθ    θ [°]   V_Rd [kN]       η
-----------------------------------
  1.00     45.0      180.71   1.328
  1.25     38.7      225.89   1.062
  1.50     33.7      271.06   0.885
  1.75     29.7      316.24   0.759
  2.00     26.6      361.42   0.664
  2.25     24.0      347.80   0.690
  2.50     21.8      323.15   0.743


## 7. Interazione torsione-taglio — NTC18 §4.1.2.3.6, [4.1.40]

Quando è presente anche un momento torcente $T_{Ed}$, la verifica di interazione è:

$$\frac{T_{Ed}}{T_{Rcd}} + \frac{V_{Ed}}{V_{Rcd}} \leq 1.0$$

**Dati torsione**: T_Ed = 15 kN·m (es. trave di bordo con solaio asimmetrico)

In [ ]:
# Sezione rettangolare — nucleo resistente a torsione
A_tot  = bw * h
u      = 2 * (bw + h)
t_eq   = A_tot / u
bw_m   = bw - t_eq
h_m    = h  - t_eq
A_k    = bw_m * h_m
u_m    = 2 * (bw_m + h_m)

phi_long_tor = 16.0
n_long_tor   = 4
Al_tor = n_long_tor * math.pi * phi_long_tor**2 / 4

print(f"t_eq = {t_eq:.1f} mm")
print(f"A_k  = {A_k:.0f} mm²")
print(f"u_m  = {u_m:.0f} mm")

T_Ed     = 15.0e6    # N·mm
V_Ed_tor = V_Ed_N

T_Rd = torsion_resistance(
    A=A_k, t=t_eq, Asw=Asw, s=s, f_yd=f_yd, f_cd=f_cd,
    sum_Al=Al_tor, um=u_m, cot_theta=cot_theta,
)

# Resistenze lato calcestruzzo per interazione [4.1.40]
nu    = 0.5
T_Rcd = 2 * A_k * t_eq * nu * f_cd * cot_theta / (1.0 + cot_theta**2)
V_Rcd = 0.9 * d * bw * nu * f_cd * cot_theta / (1.0 + cot_theta**2)

# Valori attesi — [4.1.40]: T_Ed/T_Rcd + V_Ed/V_Rcd ≤ 1.0
expected_eta_int = T_Ed / T_Rcd + V_Ed_tor / V_Rcd

eta_int = torsion_shear_interaction(T_Ed, T_Rcd, V_Ed_tor, V_Rcd)

print(f"\nT_Ed  = {T_Ed/1e6:.1f} kN·m")
print(f"T_Rd  = {T_Rd/1e6:.2f} kN·m")
print(f"T_Rcd = {T_Rcd/1e6:.2f} kN·m")
print(f"V_Rcd = {V_Rcd/1e3:.2f} kN")
print(f"\nT_Ed/T_Rcd + V_Ed/V_Rcd = {eta_int:.3f}")
print("✓  Verifica OK" if eta_int <= 1.0 else "✗  NON soddisfatta — aumentare armatura torsionale")

assert math.isclose(eta_int, expected_eta_int, rel_tol=1e-6)

## Riepilogo

In [18]:
print("=" * 55)
print("RIEPILOGO VERIFICHE A TAGLIO — NTC18 §4.1.2.3.5")
print("=" * 55)
print(f"Sezione         {bw:.0f} × {h:.0f} mm  (d = {d:.0f} mm)")
print(f"Calcestruzzo    C25/30  f_ck = {f_ck} MPa,  f_cd = {f_cd:.2f} MPa")
print(f"Acciaio         B450C   f_yk = {f_yk} MPa,  f_yd = {f_yd:.2f} MPa")
print(f"Armatura long.  {n_bars}Ø{phi_l:.0f}   ρl = {rho_l:.4f}")
print(f"Staffe          Ø{phi_sw:.0f}/{s:.0f}  2 branche  Asw = {Asw:.1f} mm²")
print("-" * 55)
print(f"V_Ed                  = {V_Ed:.2f} kN")
print(f"V_Rd (senza staffe)   = {V_Rd0_kN:.2f} kN  → {'OK' if V_Ed <= V_Rd0_kN else 'INSUFFICIENTE'}")
print(f"V_Rd (Ø{phi_sw:.0f}/{s:.0f}, cotθ={cot_theta}) = {V_Rd_kN:.2f} kN  → {'OK' if V_Ed <= V_Rd_kN else 'INSUFFICIENTE'}")
print("-" * 55)
print(f"Interazione tors-tag  = {eta_int:.3f}  → {'OK' if eta_int <= 1.0 else 'INSUFFICIENTE'}")
print("=" * 55)

RIEPILOGO VERIFICHE A TAGLIO — NTC18 §4.1.2.3.5
Sezione         300 × 550 mm  (d = 490 mm)
Calcestruzzo    C25/30  f_ck = 25.0 MPa,  f_cd = 14.17 MPa
Acciaio         B450C   f_yk = 450.0 MPa,  f_yd = 391.30 MPa
Armatura long.  4Ø20   ρl = 0.0085
Staffe          Ø10/150  2 branche  Asw = 157.1 mm²
-------------------------------------------------------
V_Ed                  = 240.00 kN
V_Rd (senza staffe)   = 80.23 kN  → INSUFFICIENTE
V_Rd (Ø10/150, cotθ=2.5) = 323.15 kN  → OK
-------------------------------------------------------
Interazione tors-tag  = 1.087  → INSUFFICIENTE
